In [ ]:
# 仓库根目录加入 sys.path（本仓库自包含，不依赖外部绝对路径）
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
from torch import nn
from d2l import torch as d2l
from deepseek_tokenizer import ds_token

from motex_utils.hybrid_attention import (DenseAttention, SparseAttentionRouter,
                                           SparseAttention)
from motex_utils.moe import MoEFeedForward, MoeTopKRouter, MoeExpertFFN, SwiGLUMLP
from motex_utils.training import train_motex_ckpt_v2, predict_motex


# Motex V2 版本
Motex V2：Moe、MLA 等改进

In [ ]:
class HybridAttentionBlock(nn.Module):
    """
    混合注意力块，直接替换 GQARopeMultiHeadAttentionKVCache，接口完全一致。
    """
    def __init__(self, key_size, query_size, value_size, num_hiddens, num_heads,
                 dropout, max_seq_len, bias=False, num_kv_heads=None, top_k=8):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads if num_kv_heads is not None else num_heads

        # 软注意力通路
        self.soft_path = DenseAttention(query_size, key_size, value_size,
                                           num_hiddens, num_heads, self.num_kv_heads,
                                           dropout, max_seq_len, bias)
        # 硬注意力通路
        self.hard_path = SparseAttention(query_size, key_size, value_size,
                                           num_hiddens, num_heads, self.num_kv_heads,
                                           top_k * num_heads, max_seq_len, bias)
        # 输出投影（软硬共享）
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)
        # 可学习融合系数
        self.alpha = nn.Parameter(torch.tensor(0.5))

    def forward(self, queries, keys, values, valid_lens, state, i):
        # 确定 query_offset
        query_offset = 0
        if not self.training and state is not None and state[0] is not None and state[0][i] is not None:
            cache_k, _ = state[0][i]
            query_offset = cache_k.shape[1]

        # 软注意力
        out_soft, state = self.soft_path(queries, keys, values, valid_lens, state, i, query_offset)

        # 构建因果掩码，供硬注意力使用
        # [问题] 原版掩码只有 (S, S)，但硬通路拼接历史缓存后 scores 形状为 (…, S, S+query_offset)，
        #       形状不匹配；且硬通路原先根本不用缓存。此处让掩码对历史列全放行、对当前列施加因果三角。
        # [解决] 掩码形状 = (S, S + query_offset)：训练时 query_offset=0 保持原行为；
        #       解码时历史列可任意注意、当前列仍为因果。并把 state 传给硬通路使其能拼缓存。
        B, S, _ = queries.shape
        device = queries.device
        causal_mask = torch.zeros(S, S + query_offset, device=device)
        causal_mask[:, query_offset:] = torch.triu(torch.ones(S, S, device=device) * -1e9, diagonal=1)

        # 硬注意力（内部调用独立 Router）
        out_hard = self.hard_path(queries, keys, values, query_offset, causal_mask, state, i)

        # 融合
        alpha = torch.sigmoid(self.alpha)
        out = alpha * out_soft + (1 - alpha) * out_hard

        # 最终输出投影
        out = self.W_o(out)
        return out, state

In [ ]:
class MotexDecoderKVCacheBlock(nn.Module):
    def __init__(self, query_size, key_size, value_size, num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_kv_heads, dropout, i, num_experts, top_k, max_seq_len):
        super().__init__()
        self.i = i
        self.norm1 = nn.RMSNorm(norm_shape)
        self.attention = HybridAttentionBlock(
            key_size=key_size,
            query_size=query_size,
            value_size=value_size,
            num_hiddens=num_hiddens,
            num_heads=num_heads,
            dropout=dropout,
            max_seq_len=max_seq_len,
            num_kv_heads=num_kv_heads
        )
        self.dropout = nn.Dropout(dropout)
        self.norm2 = nn.RMSNorm(norm_shape)
        self.ffn = MoEFeedForward(num_hiddens, ffn_num_hiddens, num_experts, top_k,  dropout)
        

    def forward(self, X, state=None, valid_lens=None):
        """
        X: (batch, seq_len, num_hiddens)
        state: 外部传入的缓存列表，结构为 [ [None]*num_layers, ... ]，
               训练时传 None；推理时 state[0] 必须为长度为 num_layers 的列表。
        valid_lens: 因果掩码，推理时通常传 None
        """
        # Pre-Norm
        normed_X = self.norm1(X)
        attn_output, state = self.attention(normed_X, normed_X, normed_X, valid_lens, state, self.i)
        X = X + self.dropout(attn_output)
        normed_X2 = self.norm2(X)
        Y, aux_loss = self.ffn(normed_X2)
        Z = X + self.dropout(Y)
        return Z, state, aux_loss

In [ ]:
class MotexDecoder(nn.Module):
    def __init__(self, query_size, key_size, value_size, 
                 num_layers, num_hiddens, num_heads, norm_shape, 
                 vocabs_size, ffn_num_input, ffn_num_hiddens, num_kv_heads, num_experts, top_k, dropout, max_seq_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocabs_size, num_hiddens)
        self.blks = nn.Sequential()
        self.num_layers = num_layers
        self.num_hiddens = num_hiddens
        for i in range(num_layers):
            self.blks.add_module(
                f"{i}", MotexDecoderKVCacheBlock(query_size=query_size, key_size=key_size, 
                                     value_size=value_size, num_hiddens=num_hiddens,
                                     norm_shape=norm_shape, ffn_num_input=ffn_num_input,
                                    ffn_num_hiddens=ffn_num_hiddens, num_heads=num_heads,
                                    num_kv_heads=num_kv_heads, dropout=dropout,
                                    max_seq_len=max_seq_len, num_experts=num_experts, top_k=top_k, i=i)
            )
   
    def forward(self, tokens, valid_lens, state=None):
        X = self.token_embedding(tokens)
        aux_loss_all = 0.
        # 推理时才需要初始化缓存列表
        if state is not None and state[0] is None:
            state[0] = [None] * self.num_layers
        for blk in self.blks:
            X, state, aux_loss = blk(X, state, valid_lens)
            aux_loss_all += aux_loss
        return X, state, aux_loss_all / len(self.blks)
        

In [ ]:
class MotexModel(nn.Module):
    def __init__(self,query_size, key_size, value_size, 
                 num_layers,num_hiddens, num_heads,norm_shape, 
                 vocabs_size, ffn_num_input, ffn_num_hiddens,num_kv_heads, num_experts, top_k, dropout,max_seq_len):
        super().__init__()
        self.dense = nn.Linear(num_hiddens, vocabs_size)
        self.decoder = MotexDecoder(query_size, key_size, value_size, 
                 num_layers,num_hiddens, num_heads, norm_shape, 
                 vocabs_size, ffn_num_input, ffn_num_hiddens, 
                 num_kv_heads, num_experts, top_k, 
                 dropout, max_seq_len)
    
    def forward(self, X ,valid_lens, state=None):
        X, state, aux_loss = self.decoder(X, valid_lens,state)
        logits = self.dense(X)
        logits = logits / (self.decoder.num_hiddens ** 0.5)
        return logits, state, aux_loss

In [ ]:
devices = d2l.try_gpu()
loss = nn.CrossEntropyLoss()
batch_size, max_len, max_seq_len, lr = 16, 256, 256, 1e-4
num_steps = 512 + 20480
torch.backends.cuda.matmul.allow_tf32 = True
# 若还想加速卷积等操作，可以额外开启 cudnn 的 TF32：
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

In [ ]:
net = MotexModel(vocabs_size = ds_token.vocab_size, num_hiddens=256, norm_shape=[256],
                    ffn_num_input=256, ffn_num_hiddens=128, num_heads=8,num_kv_heads=2,
                    num_layers=16, dropout=0.2, key_size=256, query_size=256,
                    value_size=256, max_seq_len=max_seq_len, num_experts=6, top_k=2)


In [ ]:
# ============================================================
# 数据加载（placeholder）
# 数据集与数据加载代码未随本仓库分发，请自行准备数据并在此接入，
# 例如：train_iter, test_iter = my_dataloader(batch_size, max_len)
# ============================================================
train_iter = test_iter = None  # TODO: 替换为你的数据接口后再运行


In [ ]:

train_motex_ckpt_v2(
    net, loss, train_iter, test_iter,  
    ds_token.vocab_size, devices, num_steps, lr, 
    accum_steps=1
    ,ckpt_dir='./checkpoints/Motex_v2_2'
    # 
    )

In [ ]:
output = predict_motex(net, "你是谁", 128, devices, bos_token_id=ds_token.encode("<｜end▁of▁sentence｜>"), eos_token_id=ds_token.encode('<｜end▁of▁sentence｜>'))
print(output)